In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from torch import Tensor

In [3]:
random_torch=torch.rand(4,4)
print(random_torch)

tensor([[0.4162, 0.8863, 0.4045, 0.7903],
        [0.6381, 0.2900, 0.8063, 0.5123],
        [0.5322, 0.6328, 0.0829, 0.6014],
        [0.0067, 0.1752, 0.0603, 0.6276]])


In [4]:
#将输入的离散单词数字 ID，转换为高维的连续语义向量（Dense Vector），并按照 Transformer 论文要求进行数值缩放。
class TokenEmbedding(nn.Embedding):
    def __init__(self, vocab_size, d_model):
        super().__init__(vocab_size, d_model, padding_idx=1)
        self.d_model = d_model

    def forward(self, x: Tensor) -> Tensor:
        # 重写 forward，按论文要求乘以 sqrt(d_model)
        return super().forward(x) * math.sqrt(self.d_model)

In [5]:
class PositionalEmbedding(nn.Module):
    def __init__(self,d_model,max_len,device):
        super().__init__()
        self.encoding = torch.zeros(max_len,d_model,device=device)
        self.encoding.requires_grad = False
        pos = torch.arange(0, max_len, device=device).float().unsqueeze(1)    # Shape: [max_len, 1]

        _2i = torch.arange(0, d_model, step=2, device=device).float()          # Shape: [d_model // 2]
        self.encoding[:, 0::2] = torch.sin(pos / (10000 ** (_2i / d_model)))  #偶数列用sin
        self.encoding[:, 1::2] = torch.cos(pos / (10000 ** (_2i / d_model)))   #奇数列用cos

    #根据当前输入的句子实际长度（seq_len），从事先生成好的大位置编码表（self.encoding）中，截取出刚好够用的前 seq_len 行位置向量返回。
    def forward(self,x):
        batch_size,seq_len = x.size()
        return self.encoding[:seq_len,:]

In [7]:
class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, device, dropout):
        super().__init__()
        self.tok_emb = TokenEmbedding(vocab_size, d_model)
        self.pos_emb = PositionalEmbedding(d_model, max_len, device)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x): # 注意缩进：与 __init__ 对齐
        tok_emb = self.tok_emb(x)
        pos_emb = self.pos_emb(x)
        return self.dropout(tok_emb + pos_emb)



In [8]:
#MHA 多头注意力机制
class MultiheadAttention(nn.Module):
    def __init__(self,d_model,n_heads):
        super().__init__()
        self.n_heads = n_heads
         # 1. 确保 d_model 能被 n_heads 整除（比如 512 维分成 8 个头，每个头就是 64 维）
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model //self.n_heads  # 每个头分到的维度 (例如: 512 // 8 = 64)
        self.w_q = nn.Linear(d_model,d_model)
        self.w_k = nn.Linear(d_model,d_model)
        self.w_v = nn.Linear(d_model,d_model)
        self.w_o = nn.Linear(d_model,d_model)
        self.softmax = nn.Softmax(dim= -1)

    def forward(self,q,k,v,mask=None):
        batch_size = q.size(0)
        #步骤1:通过线性层生成Q K V
        Q = self.w_q(q)
        K =self.w_k(k)
        V =self.w_v(v)
          # -------------------------------------------------------------
        # 步骤 2：拆分多头 (Reshape + Transpose)
        # 将 d_model 拆成 n_heads × head_dim，并将 n_heads 维度提到前面
        # 形状变化：[batch_size, seq_len, d_model]
        #       -> [batch_size, seq_len, n_heads, head_dim]
        #       -> [batch_size, n_heads, seq_len, head_dim]
        # -------------------------------------------------------------
        #在执行 .view() 的时候，刚从线性层 self.w_q(q) 出来的张量 Q 在物理内存中是完全连续存储的。
        Q=Q.view(batch_size,-1,self.n_heads,self.head_dim).transpose(1,2)
        K=K.view(batch_size,-1,self.n_heads,self.head_dim).transpose(1,2)
        V=V.view(batch_size,-1,self.n_heads,self.head_dim).transpose(1,2)
        scores = torch.matmul(Q,K.transpose(2,3))/math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask==0,value=-1e-9)
        attn_weights = self.softmax(scores)
        context = torch.matmul(attn_weights,V)
        # 把调换的维度调回来: [batch_size, seq_len_q, n_heads, head_dim]
        # 【关键】因为调用了 transpose，内存变得不连续，必须加 .contiguous() 才能进行 .view()
        # 把多头重新合并成 d_model 维: [batch_size, seq_len_q, d_model]
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
          # -------------------------------------------------------------
        # 步骤 7：通过最终的线性层 W_o 进行多头特征融合输出
        # output 形状：[batch_size, seq_len_q, d_model]
        # -------------------------------------------------------------
        output = self.w_o(context)
        return output
















In [9]:
#test MHA
# 模拟测试：batch_size=2, 句子长度 seq_len=5, d_model=512, 8个头
mha = MultiheadAttention(d_model=512, n_heads=8)

# 模拟输入张量 x (batch_size=2, seq_len=5, d_model=512)
x = torch.randn(2, 5, 512)

# 自注意力模式：q, k, v 都传 x
output = mha(q=x, k=x, v=x)

print("输入形状:", x.shape)
print("输出形状:", output.shape)  # 期望输出: torch.Size([2, 5, 512])

输入形状: torch.Size([2, 5, 512])
输出形状: torch.Size([2, 5, 512])


In [10]:
#LayerNorm
class LayerNorm(nn.Module):
    def __init__(self,d_model,eps=10e-12):
        super(LayerNorm,self).__init__()
        # 1. 可学习的缩放参数 gamma，初始化全为 1（与词向量维度 d_model 相同）
        # nn.Parameter 告诉 PyTorch：这是一个需要通过反向传播不断更新优化的模型参数！
        self.gamma = nn.Parameter(torch.ones(d_model))
          # 2. 可学习的平移参数 beta，初始化全为 0
        self.beta = nn.Parameter(torch.zeros(d_model))
          # 3. 防止分母为 0 的极小常数 (epsilon)，例如 1e-12
        self.eps = eps

    def forward(self, x):
        # -------------------------------------------------------------
        # 步骤 1：计算最后一个维度 (d_model) 的均值 (Mean)
        # keepdim=True 保持维度不缩减，方便后续利用广播机制做减法
        # 输入 x 形状: [batch_size, seq_len, d_model]
        # mean 形状:   [batch_size, seq_len, 1]
        # -------------------------------------------------------------
        mean = x.mean(-1, keepdim=True)
        # -------------------------------------------------------------
        # 步骤 2：计算最后一个维度 (d_model) 的方差 (Variance)
        # unbiased=False 代表使用总体方差公式（除以 N 而不是 N-1）
        # var 形状: [batch_size, seq_len, 1]
        # -------------------------------------------------------------
        var = x.var(-1, unbiased=False, keepdim=True)
        # -------------------------------------------------------------
        # 步骤 3：标准化 (Standardization)
        # 数学公式：(x - mean) / sqrt(var + eps)
        # 这一步把数据强制拉回到 均值为 0，方差为 1 的标准正态分布
        # x_norm 形状: [batch_size, seq_len, d_model]
        # -------------------------------------------------------------
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        # -------------------------------------------------------------
        # 步骤 4：缩放与平移 (Scale and Shift)
        # 数学公式：gamma * x_norm + beta
        # 作用：给予模型自我恢复原表达能力的空间，防止归一化把特征表达能力限制死
        # out 形状: [batch_size, seq_len, d_model]
        # -------------------------------------------------------------
        out = self.gamma * x_norm + self.beta
        return out






In [13]:
#FFN 前馈神经网络
# 前馈神经网络 (Feed-Forward Network)
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        # 用 nn.Sequential 自动按照顺序打包网络层
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        # forward 变成了极致优雅的一行！
        return self.ffn(x)

In [14]:
#编码器层
class EncoderLayer(nn.Module):
    def __init__(self,d_model,n_heads,d_ff,dropout=0.1):
        super().__init__()
        #1.多头注意力机制
        self.mha =MultiheadAttention(d_model,n_heads)
           # 2. 前馈神经网络
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        #两个LayerNorm层(对应两次ADD&Norm)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        #两个Dropoout层
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self,x,mask=None):

        attn_output = self.mha(x,x,x,mask)
        x1 = self.norm1(x+self.dropout1(attn_output))
        ffn_output = self.ffn(x1)
        x2 = self.norm2(x1+self.dropout2(ffn_output))
        return x2














In [15]:
#Encoder编码器整体:一个Embedding层+N个重复堆叠的EncoderLayer组成的整体
class Encoder(nn.Module):
    def __init__(self,vocab_size,d_model,n_heads,d_ff,num_layers,max_len,device,dropout=0.1):
        super().__init__()
        self.emb = TransformerEmbedding(vocab_size,d_model,max_len,device,dropout)
        self.layers = nn.ModuleList(
            [
                EncoderLayer(d_model,n_heads,d_ff,dropout) for _ in range(num_layers)
            ]
        )

    def forward(self,x,mask=None):
        #将输入的词ID转换为带有位置信息的Embedding向量
        out = self.emb(x)
        #依次穿过N个EncoderLayer进行深度特征提取
        for layer in self.layers:
            out = layer(out,mask)

        return out





In [16]:
class DecoderLayer(nn.Module):
    def __init__(self,d_model,n_heads,d_ff,dropout=0.1):
        super().__init__()
        self.msa =MultiheadAttention(d_model,n_heads)
        self.cross_attn = MultiheadAttention(d_model,n_heads)
        self.ffn =PositionwiseFeedForward(d_model,d_ff,dropout)
        self.norm1 = LayerNorm(d_model)
        self.norm2 =LayerNorm(d_model)
        self.norm3 =LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self,dec_x,enc_out,tgt_mask=None,src_mask=None):
         """
        参数说明：
        dec_x: 解码器当前的输入 Tensor [batch_size, tgt_len, d_model]
        enc_out: 编码器 Encoder 的最终输出 Tensor [batch_size, src_len, d_model]
        tgt_mask: 目标序列掩码 (Causal Mask 因果掩码)，防止偷看未来词
        src_mask: 源序列掩码 (Padding Mask)，屏蔽编码器填充符号
        """
         #子层1:掩码自注意力
         self_attn_out = self.msa(q=dec_x,k=dec_x,v=dec_x,mask=tgt_mask)
         x1 = self.norm1(dec_x+self.dropout1(self_attn_out))
         #子层2:交叉注意力层,Q 来自解码器上一层的输出 x1, K, V 来自编码器 Encoder 的输出 enc_out
         cross_attn_out = self.cross_attn(q=x1,k=enc_out,v=enc_out,mask=src_mask)
         x2= self.norm2(x1+self.dropout2(cross_attn_out))
         #子层3:FFN
         ffn_out = self.ffn(x2)
         x3 = self.norm3(x2+self.dropout3(ffn_out))
         return x3






In [18]:
class Decoder(nn.Module):
    def __init__(self,vocab_size,d_model,n_heads,d_ff,num_layers,device,max_len,dropout=0.1):
        super().__init__()

        self.emb = TransformerEmbedding(vocab_size,d_model,max_len,device,dropout)
        self.layers =nn.ModuleList(
            [
                DecoderLayer(d_model,n_heads,d_ff,dropout) for _ in range(num_layers)
            ]
        )

    def forward(self,x,enc_out,tgt_mask=None,src_mask = None):
        out = self.emb(x)
        for layer in self.layers:
            out = layer(out,enc_out,tgt_mask,src_mask)
        return out







In [19]:
class Transformermask:
    def __init__(self,pad_idx=1):
        self.pad_idx = pad_idx     #记录填充符（Padding Token）在词表中的索引数字，默认值为 1。


    def make_src_mask(self,src):
        src_mask = (src !=self.pad_idx).unsqueeze(1).unsqueeze(2)    #src_mask形状[batch_size, 1, 1, src_len]。
        return src_mask

    def make_tgt_mask(self,tgt):
        tgt_len = tgt.size(1)
        tgt_pad_mask = (tgt!=self.pad_idx).unsqueeze(1).unsqueeze(2)    #tgt_pad_mask 的形状扩充后为：[batch_size, 1, 1, tgt_len]。
        tgt_sub_mask = torch.tril(torch.ones(tgt_len,tgt_len),device=tgt.device).bool()
          # 3. 按位与结合
        tgt_mask = tgt_pad_mask & tgt_sub_mask
        return tgt_mask


In [ ]:
# Transformer 顶层主模型封装 (组合 Encoder + Decoder + Projection 线性输出层)
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, n_heads=8, d_ff=2048, 
                 num_layers=6, max_len=5000, pad_idx=1, device="cuda", dropout=0.1):
        super().__init__()
        # 1. Mask 构造器
        self.mask_builder = Transformermask(pad_idx=pad_idx)
        # 2. 编码器 Encoder
        self.encoder = Encoder(src_vocab_size, d_model, n_heads, d_ff, num_layers, max_len, device, dropout)
        # 3. 解码器 Decoder
        self.decoder = Decoder(tgt_vocab_size, d_model, n_heads, d_ff, num_layers, device, max_len, dropout)
        # 4. 输出概率映射线性层 (将 d_model 映射到目标词表大小 tgt_vocab_size)
        self.projection = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):
        # 步骤 1: 自动生成 Padding Mask 与 Causal Mask
        src_mask = self.mask_builder.make_src_mask(src)
        tgt_mask = self.mask_builder.make_tgt_mask(tgt)
        # 步骤 2: Encoder 前向传播特征提取
        enc_out = self.encoder(src, src_mask)
        # 步骤 3: Decoder 前向传播 Cross-Attention 交叉解码
        dec_out = self.decoder(tgt, enc_out, tgt_mask, src_mask)
        # 步骤 4: 映射到词表概率 Logits
        output = self.projection(dec_out)
        return output


In [ ]:
# 测试 Transformer 整体端到端前向传播
transformer = Transformer(src_vocab_size=10000, tgt_vocab_size=10000, device="cuda").cuda()

# 模拟输入张量：batch_size=2, 源句子长=5, 目标句子长=6
src = torch.randint(2, 10000, (2, 5)).cuda()
tgt = torch.randint(2, 10000, (2, 6)).cuda()

output = transformer(src, tgt)

print("输入源序列 Shape:", src.shape)                  # [2, 5]
print("输入目标序列 Shape:", tgt.shape)               # [2, 6]
print("Transformer 模型最终输出 Logits Shape:", output.shape) # 期望输出: [2, 6, 10000]
